# 🧹 Denoising Autoencoder on MNIST

This notebook builds a **Convolutional Denoising Autoencoder** using PyTorch to remove noise from handwritten digit images (MNIST).

### Pipeline
1. Load MNIST and add synthetic Gaussian noise
2. Build a Conv Autoencoder (Encoder → Bottleneck → Decoder)
3. Train to reconstruct clean images from noisy inputs
4. Visualise results side-by-side

## 1. Install & Import Dependencies

In [ ]:
# Uncomment if running for the first time
# !pip install torch torchvision matplotlib

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

## 2. Hyperparameters

In [ ]:
BATCH_SIZE   = 256
EPOCHS       = 20
LEARNING_RATE = 1e-3
NOISE_FACTOR = 0.4   # std of Gaussian noise added to inputs

## 3. Load MNIST Dataset

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),          # [0,255] → [0,1]
])

train_dataset = torchvision.datasets.MNIST(
    root='./data', train=True,  download=True, transform=transform)
test_dataset  = torchvision.datasets.MNIST(
    root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f'Train samples: {len(train_dataset)} | Test samples: {len(test_dataset)}')

## 4. Add Noise Helper

In [ ]:
def add_noise(images: torch.Tensor, noise_factor: float = NOISE_FACTOR) -> torch.Tensor:
    """Add Gaussian noise and clamp to [0, 1]."""
    noise = torch.randn_like(images) * noise_factor
    return torch.clamp(images + noise, 0.0, 1.0)

## 5. Visualise Noisy vs Clean Samples

In [ ]:
examples = next(iter(test_loader))[0][:8]
noisy_examples = add_noise(examples)

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i in range(8):
    axes[0, i].imshow(examples[i].squeeze(), cmap='gray')
    axes[0, i].axis('off')
    axes[0, i].set_title('Clean', fontsize=8)

    axes[1, i].imshow(noisy_examples[i].squeeze(), cmap='gray')
    axes[1, i].axis('off')
    axes[1, i].set_title('Noisy', fontsize=8)

plt.suptitle('Sample MNIST Images — Clean vs Noisy', fontsize=12)
plt.tight_layout()
plt.show()

## 6. Build the Convolutional Autoencoder

```
Input (1×28×28)
  │
  ▼  ENCODER
Conv 32  → BN → ReLU   (28×28)
Conv 64  → BN → ReLU   (14×14, stride 2)
Conv 128 → BN → ReLU   (7×7,   stride 2)
  │
  ▼  DECODER
ConvT 64 → BN → ReLU   (14×14)
ConvT 32 → BN → ReLU   (28×28)
ConvT 1  → Sigmoid      (28×28)
```

In [ ]:
class DenoisingAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()

        # ── Encoder ──────────────────────────────────────────────────────────
        self.encoder = nn.Sequential(
            # Block 1 — keep spatial size
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            # Block 2 — downsample 28→14
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),

            # Block 3 — downsample 14→7
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
        )

        # ── Decoder ──────────────────────────────────────────────────────────
        self.decoder = nn.Sequential(
            # Block 1 — upsample 7→14
            nn.ConvTranspose2d(128, 64, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),

            # Block 2 — upsample 14→28
            nn.ConvTranspose2d(64, 32, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            # Output — keep spatial size, compress to 1 channel
            nn.ConvTranspose2d(32, 1, kernel_size=3, padding=1),
            nn.Sigmoid(),  # output in [0, 1]
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))


model = DenoisingAutoencoder().to(DEVICE)
print(model)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nTrainable parameters: {total_params:,}')

## 7. Loss & Optimiser

In [ ]:
criterion = nn.MSELoss()                          # pixel-wise reconstruction loss
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3, verbose=True)

## 8. Training Loop

In [ ]:
train_losses, val_losses = [], []

for epoch in range(1, EPOCHS + 1):
    # ── Train ────────────────────────────────────────────────────────────────
    model.train()
    running_loss = 0.0
    for clean_imgs, _ in train_loader:
        clean_imgs = clean_imgs.to(DEVICE)
        noisy_imgs = add_noise(clean_imgs)

        optimizer.zero_grad()
        reconstructed = model(noisy_imgs)
        loss = criterion(reconstructed, clean_imgs)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    train_loss = running_loss / len(train_loader)
    train_losses.append(train_loss)

    # ── Validate ─────────────────────────────────────────────────────────────
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for clean_imgs, _ in test_loader:
            clean_imgs = clean_imgs.to(DEVICE)
            noisy_imgs = add_noise(clean_imgs)
            reconstructed = model(noisy_imgs)
            val_loss += criterion(reconstructed, clean_imgs).item()

    val_loss /= len(test_loader)
    val_losses.append(val_loss)
    scheduler.step(val_loss)

    print(f'Epoch [{epoch:02d}/{EPOCHS}]  '
          f'Train Loss: {train_loss:.5f}  |  Val Loss: {val_loss:.5f}')

print('\nTraining complete!')

## 9. Plot Training & Validation Loss

In [ ]:
plt.figure(figsize=(9, 4))
plt.plot(range(1, EPOCHS + 1), train_losses, label='Train Loss', marker='o', markersize=4)
plt.plot(range(1, EPOCHS + 1), val_losses,   label='Val Loss',   marker='s', markersize=4)
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Denoising Autoencoder — Training Curves')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Evaluate — Visualise Denoising Results

In [ ]:
model.eval()
test_images, _ = next(iter(test_loader))
test_images = test_images[:10].to(DEVICE)
noisy_images = add_noise(test_images)

with torch.no_grad():
    denoised = model(noisy_images)

# Move to CPU for plotting
test_images  = test_images.cpu()
noisy_images = noisy_images.cpu()
denoised     = denoised.cpu()

fig, axes = plt.subplots(3, 10, figsize=(16, 5))
row_labels = ['Clean', 'Noisy', 'Denoised']

for i in range(10):
    for row, imgs in enumerate([test_images, noisy_images, denoised]):
        axes[row, i].imshow(imgs[i].squeeze(), cmap='gray', vmin=0, vmax=1)
        axes[row, i].axis('off')
    axes[0, i].set_title(f'#{i+1}', fontsize=8)

for row, label in enumerate(row_labels):
    axes[row, 0].set_ylabel(label, fontsize=11, rotation=90, labelpad=40)

plt.suptitle('Denoising Autoencoder Results', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 11. Compute PSNR (Peak Signal-to-Noise Ratio)

In [ ]:
def psnr(original: torch.Tensor, reconstructed: torch.Tensor) -> float:
    """Compute average PSNR over a batch. Higher is better (dB)."""
    mse = torch.mean((original - reconstructed) ** 2, dim=[1, 2, 3])
    return torch.mean(10 * torch.log10(1.0 / (mse + 1e-8))).item()

model.eval()
all_clean, all_noisy, all_denoised = [], [], []

with torch.no_grad():
    for clean, _ in test_loader:
        clean  = clean.to(DEVICE)
        noisy  = add_noise(clean)
        output = model(noisy)
        all_clean.append(clean.cpu())
        all_noisy.append(noisy.cpu())
        all_denoised.append(output.cpu())

all_clean    = torch.cat(all_clean)
all_noisy    = torch.cat(all_noisy)
all_denoised = torch.cat(all_denoised)

psnr_noisy    = psnr(all_clean, all_noisy)
psnr_denoised = psnr(all_clean, all_denoised)

print(f'PSNR — Noisy vs Clean   : {psnr_noisy:.2f} dB')
print(f'PSNR — Denoised vs Clean: {psnr_denoised:.2f} dB')
print(f'Improvement             : +{psnr_denoised - psnr_noisy:.2f} dB')

## 12. Save & Load the Model

In [ ]:
# Save
torch.save(model.state_dict(), 'denoising_autoencoder.pth')
print('Model saved to denoising_autoencoder.pth')

# Load (example)
# loaded_model = DenoisingAutoencoder().to(DEVICE)
# loaded_model.load_state_dict(torch.load('denoising_autoencoder.pth'))
# loaded_model.eval()

## 13. Experiment Ideas

| Idea | How to try |
|---|---|
| **More noise** | Increase `NOISE_FACTOR` (e.g. 0.6) |
| **Salt-and-pepper noise** | Replace `add_noise` with a random pixel-zeroing function |
| **Deeper bottleneck** | Add more Conv layers or reduce stride channels |
| **VAE** | Replace the bottleneck with a mean/log-var reparameterisation |
| **Skip connections** | Add U-Net-style shortcuts from encoder to decoder |
| **SSIM loss** | Combine MSE with structural similarity for sharper edges |